# Реакции пользователя

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit # для разделения с учетом групп
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from solution_utils import(read_dialogs, read_labels, get_groups, clean_normalize_dialog, calculate_metrics, SEED)

In [2]:
DATA_PATH = "hw1_train.jsonl"
LABELS = np.array(["positive", "negative", "neutral"])

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))
reactions = read_labels(DATA_PATH, "reaction")
labeled_dialogs = [d for d in dialogs if reactions.get(d.get('dialog_id')) is not None]
print("размеченных диалогов", len(labeled_dialogs))

диалогов: 10000
размеченных диалогов 3000


In [3]:
user_texts = [clean_normalize_dialog(d) for d in labeled_dialogs]
n_gr, groups_ids = get_groups(user_texts)
groups_ids = np.array(groups_ids)
print(f"групп: {n_gr}")
sizes = np.bincount(groups_ids)
print(f"групп >1: {(sizes > 1).sum()}")
print(f"макс размер: {sizes.max()}")
print(f"диалогов в группах >1: {sizes[sizes > 1].sum()}")

групп: 2978
групп >1: 17
макс размер: 4
диалогов в группах >1: 39


Разделяем на train val и test с учетом баланса классов и группировки

In [4]:
X = np.array(user_texts)
y = np.array([reactions[d["dialog_id"]] for d in labeled_dialogs])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_val, idx_test = next(gss.split(X, y, groups_ids))
gss = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=SEED)
idx_train, idx_val = next(gss.split(X[idx_train_val], y[idx_train_val], groups_ids[idx_train_val]))

X_train, y_train = X[idx_train_val[idx_train]], y[idx_train_val[idx_train]] # 60%
X_val, y_val = X[idx_train_val[idx_val]], y[idx_train_val[idx_val]] # 20%
X_test, y_test = X[idx_test], y[idx_test] # 20%

In [5]:
for name, labels_part in [
    ("train", y_train),
    ("validation", y_val),
    ("test", y_test),
]:
    print(f"\n{name}: {len(labels_part)} примеров")
    print(
        pd.Series(labels_part)
        .value_counts(normalize=True)
        .reindex(LABELS, fill_value=0)
        .round(3)
    )


train: 1804 примеров
positive    0.099
negative    0.318
neutral     0.583
Name: proportion, dtype: float64

validation: 599 примеров
positive    0.114
negative    0.302
neutral     0.584
Name: proportion, dtype: float64

test: 597 примеров
positive    0.094
negative    0.281
neutral     0.625
Name: proportion, dtype: float64


Получаем Tf-idf для всех данных

In [6]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # добавляем биграммы
    sublinear_tf=True,     # уменьшает скорость роста влияния слова от его частоты
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

## Базовые модели

### Наивный Байес

Мы на семинаре использовали MultinomialNB, но сейчас больше подойдет ComplementNB, так как он специально спроектирован для несбалансированных данных, как у нас

In [7]:
alphas = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 0.9, 1.0, 1.5]
metrics_NB = []
for alpha in alphas:
    nb = ComplementNB(alpha=alpha)
    nb.fit(X_train_tfidf, y_train)
    y_pred = nb.predict(X_val_tfidf)
    metrics_NB.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_NB_df = pd.DataFrame(metrics_NB, index=alphas)
metrics_NB_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.001,0.621035,0.437770,0.428598,0.588169,"[[2, 18, 48], [7, 85, 89], [10, 55, 285]]","{'precision_positive': 0.10526315789473684, 'r..."
0.005,0.627713,0.449086,0.441967,0.597119,"[[3, 19, 46], [7, 89, 85], [9, 57, 284]]","{'precision_positive': 0.15789473684210525, 'r..."
0.010,0.631052,0.450991,0.443557,0.598748,"[[3, 19, 46], [6, 89, 86], [7, 57, 286]]","{'precision_positive': 0.1875, 'recall_positiv..."
0.020,0.634391,0.447168,0.436982,0.597214,"[[2, 17, 49], [4, 87, 90], [6, 53, 291]]","{'precision_positive': 0.16666666666666666, 'r..."
0.050,0.637730,0.438009,0.425232,0.591940,"[[1, 12, 55], [3, 79, 99], [4, 44, 302]]","{'precision_positive': 0.125, 'recall_positive..."
0.100,0.647746,0.431771,0.413350,0.588834,"[[0, 8, 60], [1, 70, 110], [0, 32, 318]]","{'precision_positive': 0.0, 'recall_positive':..."
0.200,0.647746,0.424657,0.405263,0.581773,"[[0, 5, 63], [0, 62, 119], [0, 24, 326]]","{'precision_positive': 0.0, 'recall_positive':..."
0.500,0.647746,0.418432,0.397505,0.575089,"[[0, 2, 66], [0, 55, 126], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."
0.900,0.619366,0.387124,0.355322,0.532757,"[[0, 2, 66], [0, 38, 143], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."
1.000,0.622705,0.389029,0.357158,0.535213,"[[0, 2, 66], [0, 38, 143], [0, 15, 335]]","{'precision_positive': 0.0, 'recall_positive':..."


По всем параметрам, кроме обычного accuracy, лучше всего alpha = 0.01, выбираем его

In [8]:
alpha_opt = 0.01
nb = ComplementNB(alpha=alpha_opt).fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_val_tfidf)

## Логистическая регрессия

In [9]:
C_grid = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 1.5, 2.0, 5.0]

metrics_logreg = []
for C in C_grid:
    log_reg = LogisticRegression(
        class_weight="balanced", # веса классов обратно пропорционально частоте
        max_iter=5000,
        C=C,
    )
    log_reg.fit(X_train_tfidf, y_train)
    y_pred = log_reg.predict(X_val_tfidf)
    metrics_logreg.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_logreg_df = pd.DataFrame(metrics_logreg, index=C_grid)
metrics_logreg_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.001,0.639399,0.561606,0.565564,0.638219,"[[28, 14, 26], [14, 97, 70], [20, 72, 258]]","{'precision_positive': 0.45161290322580644, 'r..."
0.010,0.637730,0.556704,0.561162,0.636231,"[[27, 14, 27], [14, 97, 70], [20, 72, 258]]","{'precision_positive': 0.4426229508196721, 're..."
0.050,0.642738,0.558672,0.565173,0.640056,"[[27, 14, 27], [13, 96, 72], [19, 69, 262]]","{'precision_positive': 0.4576271186440678, 're..."
0.100,0.646077,0.560577,0.570273,0.642485,"[[27, 14, 27], [11, 96, 74], [17, 69, 264]]","{'precision_positive': 0.4909090909090909, 're..."
0.500,0.651085,0.536180,0.554246,0.640109,"[[21, 13, 34], [7, 92, 82], [13, 60, 277]]","{'precision_positive': 0.5121951219512195, 're..."
1.000,0.656093,0.524128,0.546620,0.640213,"[[17, 14, 37], [3, 93, 85], [8, 59, 283]]","{'precision_positive': 0.6071428571428571, 're..."
1.500,0.656093,0.523239,0.545943,0.639787,"[[17, 14, 37], [3, 92, 86], [8, 58, 284]]","{'precision_positive': 0.6071428571428571, 're..."
2.000,0.659432,0.522083,0.545283,0.641739,"[[16, 14, 38], [2, 93, 86], [7, 57, 286]]","{'precision_positive': 0.64, 'recall_positive'..."
5.000,0.656093,0.511783,0.535763,0.635311,"[[15, 14, 39], [2, 88, 91], [5, 55, 290]]","{'precision_positive': 0.6818181818181818, 're..."


Выбираем C = 0.1

In [10]:
C_opt_logreg = 0.1
log_reg = LogisticRegression(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_logreg,
).fit(X_train_tfidf, y_train)
y_pred_logreg = log_reg.predict(X_val_tfidf)

## Метод опорных векторов

In [11]:
metrics_svc = []
for C in C_grid:
    svc = LinearSVC(
        class_weight="balanced", # веса классов обратно пропорционально частоте
        max_iter=5000,
        C=C,
    )
    svc.fit(X_train_tfidf, y_train)
    y_pred = svc.predict(X_val_tfidf)
    metrics_svc.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_svc_df = pd.DataFrame(metrics_svc, index=C_grid)
metrics_svc_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.001,0.584307,0.333333,0.245873,0.430996,"[[0, 0, 68], [0, 0, 181], [0, 0, 350]]","{'precision_positive': 0.0, 'recall_positive':..."
0.010,0.584307,0.333333,0.245873,0.430996,"[[0, 0, 68], [0, 0, 181], [0, 0, 350]]","{'precision_positive': 0.0, 'recall_positive':..."
0.050,0.662771,0.438067,0.423935,0.597326,"[[1, 8, 59], [0, 63, 118], [0, 17, 333]]","{'precision_positive': 1.0, 'recall_positive':..."
0.100,0.659432,0.461742,0.465887,0.612450,"[[5, 9, 54], [0, 74, 107], [1, 33, 316]]","{'precision_positive': 0.8333333333333334, 're..."
0.500,0.664441,0.499360,0.521174,0.634686,"[[12, 12, 44], [1, 82, 98], [4, 42, 304]]","{'precision_positive': 0.7058823529411765, 're..."
1.000,0.669449,0.504885,0.525771,0.640835,"[[12, 13, 43], [2, 85, 94], [4, 42, 304]]","{'precision_positive': 0.6666666666666666, 're..."
1.500,0.669449,0.500936,0.519768,0.639377,"[[11, 13, 44], [2, 85, 94], [4, 41, 305]]","{'precision_positive': 0.6470588235294118, 're..."
2.000,0.671119,0.501888,0.520609,0.640851,"[[11, 14, 43], [2, 85, 94], [4, 40, 306]]","{'precision_positive': 0.6470588235294118, 're..."
5.000,0.666110,0.498142,0.516622,0.636087,"[[11, 14, 43], [2, 84, 95], [4, 42, 304]]","{'precision_positive': 0.6470588235294118, 're..."


Лучший С = 1

In [12]:
C_opt_svc = 1
svc = LinearSVC(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_svc,
).fit(X_train_tfidf, y_train)
y_pred_svc = svc.predict(X_val_tfidf)

Изучим, как влияет выбор только первых/последних реплик на предсказания. Предположения: основная информация о реакции пользователя содержится в конце.

In [13]:
texts_by_variant = {
    "all": [
        clean_normalize_dialog(d)
        for d in labeled_dialogs
    ],
    "first": [
        clean_normalize_dialog(d, num=1, position='first')
        for d in labeled_dialogs
    ],
    "last": [
        clean_normalize_dialog(d, num=1, position='last')
        for d in labeled_dialogs
    ],
    "last_two": [
        clean_normalize_dialog(d, num=2, position='last')
        for d in labeled_dialogs
    ],
}

In [14]:
variant_results = []
variant_models = {}

for variant, texts in texts_by_variant.items():
    # Используем один и тот же split для всех вариантов
    X_train_text = [texts[i] for i in idx_train_val[idx_train]]
    X_val_text = [texts[i] for i in idx_train_val[idx_val]]
    
    vectorizer = TfidfVectorizer(
      ngram_range=(1, 2),
      sublinear_tf=True,  
    )

    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_val_vec = vectorizer.transform(X_val_text)

    model = LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=5000,
        random_state=SEED,
    )

    model.fit(X_train_vec, y_train)
    pred = model.predict(X_val_vec)

    variant_results.append({
        "variant": variant,
        "macro_f1": f1_score(
            y_val,
            pred,
            labels=LABELS,
            average="macro",
            zero_division=0,
        ),
    })

    variant_models[variant] = {
        "vectorizer": vectorizer,
        "model": model,
    }

variant_results_df = (
    pd.DataFrame(variant_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(variant_results_df)

,variant,macro_f1
0,last_two,0.586453
1,all,0.570273
2,first,0.498874
3,last,0.484738


In [ ]:
#TODO добавить в качестве признака количество важных знаков препинания !!! ???
# проверить влияние данных вроде длины реплик, попробовать разные уровни очистки текста и нормализации
# добавить в работу с кодом выкидывание по ''' !!!!!!